In [ ]:
#  Data Collection

!pip install arxiv requests tqdm python-dateutil -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 2.7 MB/s eta 0:00:00


In [ ]:

import arxiv
import json
import os
from pathlib import Path
from tqdm import tqdm
import time
from datetime import datetime, timedelta
import requests

print(f"Data will be saved to: {DATA_DIR.absolute()}")

Data will be saved to: /content/drive/MyDrive/multi_agent_rag_project/data/raw/arxiv_papers


In [ ]:

CONFIG = {
    'categories': ['cs.LG', 'cs.AI', 'cs.CL', 'cs.CV'],
    'max_papers_per_category': 2500,
    'start_date': '2023-01-01',
    'end_date': '2025-01-01',
    'download_pdfs': True,
    'max_retries': 3,
    'rate_limit_delay': 3
}

print("Configuration:")
for key, value in CONFIG.items():
    print(f"  {key}: {value}")


Configuration:
  categories: ['cs.LG', 'cs.AI', 'cs.CL', 'cs.CV']
  max_papers_per_category: 2500
  start_date: 2023-01-01
  end_date: 2025-01-01
  download_pdfs: True
  max_retries: 3
  rate_limit_delay: 3


In [ ]:
def search_arxiv_papers(category, max_results=2500, start_date='2023-01-01', end_date='2025-01-01'):
    """
    Search ArXiv for papers in a specific category within date range.
    """
    # Convert dates for ArXiv query format
    from datetime import datetime
    start_dt = datetime.strptime(start_date, '%Y-%m-%d')
    end_dt = datetime.strptime(end_date, '%Y-%m-%d')

    # Build search query with date range
    query = f"cat:{category} AND submittedDate:[{start_dt.strftime('%Y%m%d')}0000 TO {end_dt.strftime('%Y%m%d')}2359]"

    # Create client with proper rate limiting
    client = arxiv.Client(
        page_size=100,
        delay_seconds=3,
        num_retries=5  # Increased retries
    )

    # Search
    search = arxiv.Search(
        query=query,
        max_results=max_results,
        sort_by=arxiv.SortCriterion.SubmittedDate,
        sort_order=arxiv.SortOrder.Descending
    )

    papers = []

    print(f"\nSearching category: {category}")
    print(f"Query: {query}")

    try:
        for result in tqdm(client.results(search), total=max_results, desc=f"{category}"):
            paper_data = {
                'arxiv_id': result.entry_id.split('/')[-1],
                'title': result.title,
                'authors': [author.name for author in result.authors],
                'abstract': result.summary,
                'categories': result.categories,
                'primary_category': result.primary_category,
                'published': result.published.strftime('%Y-%m-%d'),
                'updated': result.updated.strftime('%Y-%m-%d'),
                'pdf_url': result.pdf_url,
                'comment': result.comment if hasattr(result, 'comment') else None,
                'journal_ref': result.journal_ref if hasattr(result, 'journal_ref') else None,
            }

            papers.append(paper_data)

            if len(papers) >= max_results:
                break

    except Exception as e:
        print(f"\nWarning: Stopped early due to error: {e}")
        print(f"Collected {len(papers)} papers before error")

    print(f"Found {len(papers)} papers in {category}")
    return papers

In [ ]:



all_papers = []
for category in CONFIG['categories']:
    papers = search_arxiv_papers(
        category=category,
        max_results=CONFIG['max_papers_per_category'],
        start_date=CONFIG['start_date'],
        end_date=CONFIG['end_date']
    )
    all_papers.extend(papers)
    time.sleep(CONFIG['rate_limit_delay'])

print(f"\n{'='*60}")
print(f"Total papers collected: {len(all_papers)}")
print(f"{'='*60}")


Searching category: cs.LG
Query: cat:cs.LG AND submittedDate:[202301010000 TO 202501012359]


cs.LG:   8%|▊         | 200/2500 [00:23<04:30,  8.49it/s]



Collected 200 papers before error
Found 200 papers in cs.LG

Searching category: cs.AI
Query: cat:cs.AI AND submittedDate:[202301010000 TO 202501012359]


cs.AI:   4%|▍         | 100/2500 [00:19<07:38,  5.23it/s]



Collected 100 papers before error
Found 100 papers in cs.AI

Searching category: cs.CL
Query: cat:cs.CL AND submittedDate:[202301010000 TO 202501012359]


cs.CL:   8%|▊         | 200/2500 [00:26<05:03,  7.59it/s]



Collected 200 papers before error
Found 200 papers in cs.CL

Searching category: cs.CV
Query: cat:cs.CV AND submittedDate:[202301010000 TO 202501012359]


cs.CV:   8%|▊         | 200/2500 [00:27<05:13,  7.33it/s]



Collected 200 papers before error
Found 200 papers in cs.CV

Total papers collected: 700


In [ ]:

unique_papers = {}
for paper in all_papers:
    arxiv_id = paper['arxiv_id']
    if arxiv_id not in unique_papers:
        unique_papers[arxiv_id] = paper
all_papers = list(unique_papers.values())
print(f"After deduplication: {len(all_papers)} unique papers")

After deduplication: 527 unique papers


In [ ]:


metadata_path = BASE_DIR /"data/raw/arxiv_metadata.json"
metadata_path.parent.mkdir(parents=True, exist_ok=True)
with open(metadata_path, 'w') as f:
    json.dump(all_papers, f, indent=2)
print(f"Metadata saved to: {metadata_path.absolute()}")

Metadata saved to: /content/drive/MyDrive/multi_agent_rag_project/data/raw/arxiv_metadata.json


In [ ]:



if CONFIG['download_pdfs']:
    print("\nDownloading PDFs...")
    successful_downloads = 0
    failed_downloads = 0

    for paper in tqdm(all_papers, desc="Downloading PDFs"):
        success = download_pdf(paper, DATA_DIR, max_retries=CONFIG['max_retries'])
        if success:
            successful_downloads += 1
        else:
            failed_downloads += 1
        time.sleep(CONFIG['rate_limit_delay'])

    print(f"\n{'='*60}")
    print(f"Download complete!")
    print(f"Successful: {successful_downloads}")
    print(f"Failed: {failed_downloads}")
    print(f"{'='*60}")
else:
    print("\nPDF download skipped")



Download complete!
Successful: 527
Failed: 0


In [ ]:

from collections import Counter
primary_categories = [p['primary_category'] for p in all_papers]
category_counts = Counter(primary_categories)

print("\nPapers by Category:")
for cat, count in category_counts.most_common():
    print(f"  {cat}: {count}")

publication_years = [p['published'][:4] for p in all_papers]
year_counts = Counter(publication_years)

print("\nPapers by Year:")
for year in sorted(year_counts.keys()):
    print(f"  {year}: {year_counts[year]}")

print(f"\nTotal unique papers: {len(all_papers)}")
print("Notebook 01 Complete!")


Papers by Category:
  cs.CV: 162
  cs.CL: 141
  cs.LG: 94
  eess.IV: 19
  cs.AI: 19
  cs.CR: 9
  cs.SE: 7
  cs.SD: 7
  stat.ML: 6
  cs.HC: 5
  cs.CY: 5
  cs.RO: 4
  cs.NE: 4
  eess.AS: 4
  cs.IR: 3
  eess.SP: 2
  math.OC: 2
  stat.ME: 2
  cs.AR: 2
  eess.SY: 2
  q-bio.NC: 2
  cs.NI: 2
  cs.MA: 2
  cs.LO: 2
  q-fin.ST: 1
  physics.geo-ph: 1
  cs.DC: 1
  stat.CO: 1
  physics.flu-dyn: 1
  math.NA: 1
  physics.ao-ph: 1
  quant-ph: 1
  astro-ph.GA: 1
  hep-th: 1
  cs.DS: 1
  cs.IT: 1
  q-fin.TR: 1
  cs.SI: 1
  econ.GN: 1
  cs.PL: 1
  cs.DL: 1
  cs.MM: 1
  physics.chem-ph: 1
  cs.GR: 1

Papers by Year:
  2024: 405
  2025: 122

Total unique papers: 527
Notebook 01 Complete!
